In [1]:
import galois
import numpy as np


class BCH:
    def __init__(self, p, m, t):
        self.p = p
        self.m = m
        self.n = p**m - 1
        self.field_ext = galois.GF(p**m)
        self.field_sym = galois.GF(p)
        self.alpha = self.field_ext.primitive_element
        
        self.min_polys = []
        self.g = None
        self.t = t
        self.k = 0
        
        self._initialize_generator()
        self.print_info()

    def _initialize_generator(self):
        """
        Iteratively finds the largest t such that n - deg(g) >= k_target.
        """
        current_g = galois.Poly([1], field=self.field_sym)
        seen_min_polys = set()
        
        # We search for the correction capability t
        for i in range(1, self.t*2+1):
            # Get minimal poly for alpha^i
            m_poly = (self.alpha**i).minimal_poly()
            m_poly_coeffs = tuple(m_poly.coeffs.tolist())
            
            if m_poly_coeffs not in seen_min_polys:
                # Check if adding this poly exceeds our k budget
                temp_g = current_g * m_poly
                
                current_g = temp_g
                seen_min_polys.add(m_poly_coeffs)
                self.min_polys.append(m_poly)

        self.g = current_g
        self.k = self.n - self.g.degree

    def print_info(self):
        print("-" * 40)
        print(f"BCH Code Configuration (GF({self.p}^{self.m}))")
        print("-" * 40)
        print(f"n (Block Length):    {self.n}")
        print(f"k (Message Length):  {self.k}")
        print(f"t (Corrects errors): {self.t}")
        print(f"Parity Symbols:      {self.n - self.k}")
        print(f"Primitive Poly:      {self.field_ext.irreducible_poly}")
        print(f"\nUnique Minimal Polynomials used for g(x):")
        for idx, poly in enumerate(self.min_polys):
            print(f"  M_{idx+1}(x): {poly}")
        print(f"\nFinal Generator Polynomial g(x):")
        print(f"  {self.g}")
        print("-" * 40)
    def is_legal_codeword_division(self, received_word):
        r_poly = galois.Poly(received_word, field=self.field_sym)
        # self.g is your generator polynomial
        remainder = r_poly % self.g
        return remainder == 0
    def encode(self, message):
        """
        Takes a message of length k (list or array of integers 0..p-1)
        Returns a systematic codeword of length n.
        """
        if len(message) != self.k:
            raise ValueError(f"Message length must be {self.k}, but got {len(message)}")

        # Convert input to a galois.Poly over GF(p)
        msg_poly = galois.Poly(message, field=self.field_sym)
        
        # Shift message poly: m(x) * x^(n-k)
        parity_degree = self.n - self.k
        shifted_msg = msg_poly * galois.Poly.Degrees([parity_degree], field=self.field_sym)
        
        # Calculate parity: remainder of (shifted_msg / g)
        parity = shifted_msg % self.g
        
        # Systematic codeword = shifted_msg - parity
        codeword_poly = shifted_msg - parity
        
        # FIX: Access coefficients as a property and pad manually to length n
        # We pad with leading zeros because coefficients are returned from highest degree down
        coeffs = codeword_poly.coeffs
        padding_size = self.n - len(coeffs)
        codeword_coeffs = np.pad(coeffs, (padding_size, 0), 'constant', constant_values=0)
        
        return codeword_coeffs
    def inject_noise(self, codeword, num_errors=0, num_erasures=0):
        """
        Injects random errors and erasures into the codeword.
        Returns:
            corrupted_codeword: The codeword with noise.
            erasure_indices: A list of indices where erasures occurred.
        """
        # Work on a copy to avoid modifying the original
        corrupted = np.copy(codeword)
        all_indices = list(range(self.n))
        np.random.shuffle(all_indices)
        
        # 1. Select indices for erasures
        erasure_indices = sorted(all_indices[:num_erasures])
        # In a real system, an erasure value is often set to 0 or a special flag
        for idx in erasure_indices:
            # We "erase" by changing the value, but the decoder is told the index
            current_val = corrupted[idx]
            noise = np.random.randint(1, self.p)
            corrupted[idx] = (current_val + noise) % self.p
            
        # 2. Select indices for unknown errors (from remaining indices)
        error_indices = all_indices[num_erasures : num_erasures + num_errors]
        for idx in error_indices:
            current_val = corrupted[idx]
            # Ensure the error actually changes the value
            noise = np.random.randint(1, self.p)
            corrupted[idx] = (current_val + noise) % self.p
            
        return corrupted, erasure_indices, error_indices
    def calculate_syndromes(self, received_word):
        """
        Calculates the 2t syndromes of the received word.
        S_i = R(alpha^i)
        """
        # Convert received symbols to a polynomial in the extension field
        # Note: coefficients are high-to-low degree
        r_poly = galois.Poly(received_word, field=self.field_ext)
        
        syndromes = []
        # We need 2*t syndromes for the roots alpha^1 to alpha^2t
        for i in range(1, 2 * self.t + 1):
            root = self.alpha**i
            s_i = r_poly(root) # Evaluate polynomial at alpha^i
            syndromes.append(s_i)
            
        return self.field_ext(syndromes)

    def calculate_locator(self, erasure_indices):
        """
        Gamma(x) = Product of (1 - alpha^j * x) for all j in erasure_indices.
        Note: j is the index in the codeword (0 to n-1).
        Because our poly is high-to-low, index 'j' corresponds to x^(n-1-j).
        """
        gamma = galois.Poly([1], field=self.field_ext)
        for idx in erasure_indices:
            # The location in the polynomial sense is alpha^(n-1-idx)
            location = self.alpha**(self.n - 1 - idx)
            term = galois.Poly([location, -1], field=self.field_ext) # (location*x + 1)
            gamma = gamma * term
            
        return gamma
    def calculate_modified_syndromes(self, syndromes, erasure_locator):
        """
        S'(x) = S(x) * Gamma(x) mod x^(2t)
        The first deg(Gamma) syndromes are used to 'absorb' erasures.
        BMA only needs the remaining (2t - deg(Gamma)) syndromes.
        """
        # S(x) = S_1 + S_2x + ... + S_{2t}x^{2t-1}
        s_poly = galois.Poly(syndromes[::-1], field=self.field_ext)
        
        # Product S(x) * Gamma(x)
        combined_poly = s_poly * erasure_locator
        
        # In errors-and-erasures decoding, the 'useful' syndromes for 
        # finding unknown errors are S'_{rho+1} to S'_{2t}
        # where rho = deg(erasure_locator).
        rho = erasure_locator.degree
        target_len = 2 * self.t
        
        # Get all coefficients [x^(deg), ..., x^1, x^0]
        all_coeffs = combined_poly.coeffs
        
        # Pad to ensure we have at least target_len coefficients to pick from
        if len(all_coeffs) < target_len:
            all_coeffs = np.pad(all_coeffs, (target_len - len(all_coeffs), 0), 'constant')
            
        # Reverse to get [S'_1, S'_2, ..., S'_2t]
        s_modified_all = all_coeffs[::-1]
        
        # Slice from index rho to 2t
        # These are the syndromes S'_{rho+1} ... S'_{2t}
        final_syndromes = s_modified_all[rho:target_len]
        return self.field_ext(final_syndromes)
    def solve_bma(self, syndromes):
        """
        Berlekamp-Massey Algorithm for GF(p^m).
        Returns the Error Locator Polynomial Lambda(x).
        """
        t = len(syndromes)
        # Initial conditions
        lambda_poly = galois.Poly([1], field=self.field_ext)
        old_lambda = galois.Poly([1], field=self.field_ext)
        
        l = 0  # Current number of errors found
        m = 1  # Shift factor
        b = self.field_ext(1) # Previous discrepancy
        
        # Iterating through the 2t syndromes
        for n in range(t):
            # Calculate discrepancy (d)
            # d = S_{n+1} + sum_{i=1}^{L} lambda_i * S_{n+1-i}
            d = syndromes[n]
            lambda_coeffs = lambda_poly.coeffs[::-1] # low degree to high
            for i in range(1, len(lambda_coeffs)):
                if n - i >= 0:
                    d += lambda_coeffs[i] * syndromes[n - i]
            
            if d == 0:
                m += 1
            else:
                old_poly_shifted = old_lambda * galois.Poly.Degrees([m], field=self.field_ext)
                t_poly = lambda_poly - (d / b) * old_poly_shifted
                
                if 2 * l <= n:
                    old_lambda = lambda_poly
                    lambda_poly = t_poly
                    l = n + 1 - l
                    b = d
                    m = 1
                else:
                    lambda_poly = t_poly
                    m += 1
                    
        return lambda_poly
    def solve_eea(self, modified_syndromes, num_erasures):
        """
        EEA for Errors and Erasures.
        modified_syndromes: S'(x) = S(x) * Gamma(x) mod x^2t
        num_erasures: degree of Gamma(x)
        """
        # The number of syndromes we are working with is 2t
        # But we only have (2t - num_erasures) 'useful' syndromes left
        # deg_limit = 2 * self.t
        deg_limit = len(modified_syndromes)
        
        # r0 = x^(2t)
        r0 = galois.Poly.Degrees([deg_limit], field=self.field_ext)
        # r1 = S'(x). Note: modified_syndromes should be S'_1 to S'_2t
        r1 = galois.Poly(modified_syndromes[::-1], field=self.field_ext)
        
        v0 = galois.Poly([0], field=self.field_ext)
        v1 = galois.Poly([1], field=self.field_ext)
        
        # Stopping condition for errors + erasures:
        # We need to find unknown error locator Sigma(x)
        # Degree of Sigma(x) <= (2t - num_erasures) / 2
        target_degree = deg_limit // 2
        
        while r1.degree >= target_degree:
            q, r = divmod(r0, r1)
            v = v0 - q * v1
            
            r0, r1 = r1, r
            v0, v1 = v1, v
            
        # Lambda_total(x) = Sigma(x) * Gamma(x)
        # v1 here is the Sigma(x) (unknown error locator)
        # To get the final Lambda, we must multiply by erasure_locator outside
        # or pass erasure_locator in.
        
        # Normalize
        scaling_factor = v1.coeffs[-1]
        inv_scaling = self.field_ext(1) / scaling_factor
        
        sigma_poly = v1 * inv_scaling
        omega_poly = r1 * inv_scaling
        
        return sigma_poly, omega_poly
    def chien_search(self, lambda_poly):
        """
        Locates the errors by testing every possible position.
        The error locations are the indices i where Lambda(alpha^-i) = 0.
        """
        error_indices = []
        # We check every position from 0 to n-1
        # In polynomial terms, index i corresponds to alpha^(n-1-i)
        for i in range(self.n):
            # Evaluate at alpha**(-(n-1-i))
            # Mathematically equivalent to checking if alpha**(n-1-i) is a root
            inv_loc = self.alpha**(-(self.n - 1 - i))
            if lambda_poly(inv_loc) == 0:
                error_indices.append(i)
                
        return error_indices
    def derive_omega(self, lambda_poly, syndromes):
        """
        Derives Omega(x) = [Lambda(x) * S(x)] mod x^(2t)
        Works for both BMA and EEA outputs.
        """
        # 1. Convert syndrome list to a polynomial S(x) = S_1 + S_2x + ...
        s_poly = galois.Poly(syndromes[::-1], field=self.field_ext)
        
        # 2. Multiply Lambda(x) by S(x)
        combined = lambda_poly * s_poly
        
        # 3. Take modulo x^(2t)
        # We only keep terms from x^0 to x^(2t-1)
        mod_degree = 2 * self.t
        mod_poly = galois.Poly.Degrees([mod_degree], field=self.field_ext)
        _, omega_poly = divmod(combined, mod_poly)
        
        return omega_poly
    def forney_algorithm(self, omega_poly, lambda_poly, error_indices):
        """
        Calculates error magnitudes for given indices.
        error_indices: both unknown error positions and known erasures.
        """
        magnitudes = {}
        # Formal derivative of the total locator polynomial
        lambda_prime = lambda_poly.derivative()
        
        for idx in error_indices:
            # X_j^-1 is the value we plug into the polynomials
            # It is the root of the locator polynomial
            root = self.alpha**(-(self.n - 1 - idx))
            
            # Numerator: Omega(root)
            num = omega_poly(root)
            
            # Denominator: Lambda'(root)
            den = lambda_prime(root)
            
            # Magnitude Y_j = - (Omega / Lambda')
            # In Finite Fields, subtraction is addition of the additive inverse
            mag = -(num / den)
            magnitudes[idx] = mag
            
        return magnitudes
# Create the codec
bch_codec = BCH(p=3, m=4, t=5)
def product_view(bch, lambda_poly, syndromes, modified_syndromes, erasures_indices, error_indices):
    print("-" * 30)

    m_syndromes_poly = galois.Poly(modified_syndromes[::-1], field=bch.field_ext)
    syndromes_poly = galois.Poly(syndromes[::-1], field=bch.field_ext)


    error_locator = bch_codec.calculate_locator( error_indices)
    erasures_locator = bch_codec.calculate_locator( erasures_indices)
    # 2. Calculate lambda(x) * S'(x)
    lhs_gt = error_locator * m_syndromes_poly
    lhs = lambda_poly * m_syndromes_poly
    print(f"Ground truth : error_locator(x) * S'(x) coefficients: {lhs_gt.coeffs}")
    print(f"lambda(x) * S'(x) coefficients: {lhs.coeffs}")

    # 2. Calculate lambda(x) * S(x)
    lambda_gamma_syndromes_poly_gt = syndromes_poly * error_locator*erasures_locator
    lambda_gamma_syndromes_poly = syndromes_poly * lambda_poly*erasures_locator
    print(f"Ground truth : error_locator(x)*gamma(x)*S(x) coefficients: {lambda_gamma_syndromes_poly_gt.coeffs}")
    print(f"lambda(x)*gamma(x)*S(x) coefficients: {lambda_gamma_syndromes_poly.coeffs}")


    print("-" * 30)

----------------------------------------
BCH Code Configuration (GF(3^4))
----------------------------------------
n (Block Length):    80
k (Message Length):  54
t (Corrects errors): 5
Parity Symbols:      26
Primitive Poly:      x^4 + 2x^3 + 2

Unique Minimal Polynomials used for g(x):
  M_1(x): x^4 + 2x^3 + 2
  M_2(x): x^4 + 2x^3 + x^2 + 1
  M_3(x): x^4 + x^3 + 2x + 1
  M_4(x): x^4 + 2x^2 + 2
  M_5(x): x^4 + x^3 + x^2 + 2x + 2
  M_6(x): x^4 + 2x^3 + x^2 + 2x + 1
  M_7(x): x^2 + 2x + 2

Final Generator Polynomial g(x):
  x^26 + x^25 + 2x^23 + 2x^22 + x^21 + 2x^19 + x^18 + x^17 + 2x^16 + x^15 + x^13 + x^12 + 2x^11 + x^10 + 2x^8 + x^6 + 2x^5 + 2x^3 + 2x^2 + 1
----------------------------------------


In [2]:
for i in range(1000):
    # 1. Encode
    msg = [np.random.randint(0, bch_codec.p) for _ in range(bch_codec.k)]
    codeword = bch_codec.encode(msg)
    # 2. Inject Noise
    # Recall our rule: 2*errors + erasures < d_min
    num_errors = 3
    num_erasures = 3
    received, erasures, errors = bch_codec.inject_noise(codeword, num_errors=num_errors, num_erasures=num_erasures)

    # Verify that the received word is different from the codeword
    diff_count = np.sum(received != codeword)
    
    syndromes = bch_codec.calculate_syndromes(received_word=received)
    erasures_locator = bch_codec.calculate_locator(erasure_indices=erasures)
    modified_syndromes = bch_codec.calculate_modified_syndromes(syndromes=syndromes,erasure_locator=erasures_locator)
    lambda_bma = bch_codec.solve_bma(modified_syndromes)
    lambda_eea, omega_eea = bch_codec.solve_eea(modified_syndromes,num_erasures)
    error_pos = bch_codec.chien_search( lambda_bma)
    error_pos = sorted(error_pos)
    errors = sorted(errors)
    if lambda_bma != lambda_eea or error_pos!=errors:    

        print(f"Codeword: {codeword}") 
        print(f"Received: {received}")
        print(f"Erasure indices (known to decoder): {erasures}")
        print(f"Error indices (unknown to decoder): {errors}")
        print(f"Total corrupted symbols: {diff_count}")
        print(f"Syndromes is :{syndromes}")
        print(f"Erasures locator is :{erasures_locator}")
        print(f"Modified Syndromes is :{modified_syndromes}")
        print(f"BMA Lambda: {lambda_bma}")
        print(f"EEA Lambda: {lambda_eea}")
        print(f"Match? {lambda_bma == lambda_eea}")
        print(f"chien_search bma {bch_codec.chien_search( lambda_bma)}")
        print(f"chien_search eea {bch_codec.chien_search( lambda_eea)}")
        sigma_real = bch_codec.calculate_locator(errors)
        print(f"chien_search ground truth {bch_codec.chien_search( sigma_real)}")
        print(f"Ground truth error locator {sigma_real}")
        break
    # product_view(bch_codec, lambda_bma, syndromes, modified_syndromes, erasures, errors)

    # 1. Total Locator: Combine Unknown Error Locator (sigma) and Erasure Locator (gamma)
    # sigma_eea comes from your EEA solver
    total_lambda = lambda_eea * erasures_locator

    # 2. Find ALL positions (Errors + Erasures)
    # In a perfect world, this should return exactly the indices you injected
    all_error_indices = error_pos+erasures
       
    # 3. Calculate Magnitudes
    # omega_eea also comes from your EEA solver
    error_values = bch_codec.forney_algorithm(bch_codec.derive_omega(total_lambda, syndromes), total_lambda, all_error_indices)

    # 4. Correct the Received Word
    # 1. Ensure the array itself is a Galois Field Array (GF(3))
    # If it's just a numpy array, the library will throw that TypeError
    corrected_word = bch_codec.field_sym(received) 

    # 2. Iterate through and subtract
    for idx, val in error_values.items():
        # Downcast the extension field magnitude to the base field
        mag_base = bch_codec.field_sym(val)
        
        # Now both operands are GF(3) instances, and the library will be happy
        corrected_word[idx] -= mag_base

    # 3. Check results
    # Note: codeword should also be a field_sym array for a direct comparison
    if np.array_equal(corrected_word, codeword):
        print(f"Correction Successful")
    else :
        print("Error")
        break

Correction Successful
Correction Successful
Correction Successful
Correction Successful
Correction Successful
Correction Successful
Correction Successful
Correction Successful
Correction Successful
Correction Successful
Correction Successful
Correction Successful
Correction Successful
Correction Successful
Correction Successful
Correction Successful
Correction Successful
Correction Successful
Correction Successful
Correction Successful
Correction Successful
Correction Successful
Correction Successful
Correction Successful
Correction Successful
Correction Successful
Correction Successful
Correction Successful
Correction Successful
Correction Successful
Correction Successful
Correction Successful
Correction Successful
Correction Successful
Correction Successful
Correction Successful
Correction Successful
Correction Successful
Correction Successful
Correction Successful
Correction Successful
Correction Successful
Correction Successful
Correction Successful
Correction Successful
Correction